# C2: Learning from labels (Strands head-only, LoRA, LoRA + head)

A 2 × 2 design on the joint question with probability features only: zero-shot (B1 Strands joint, no new run), head-only (the released pointer head retrained on cached C1 `<answer>` and option-end vectors, torso frozen), LoRA only (the released adapter trained, head frozen) and LoRA + head. Every trained cell starts from Strands' released weights and uses its published recipe (rank-16/alpha-32 LoRA per soup member on attention, MLP and recurrent projections; LoRA 1e-4, head 1e-3; weight decay 0.01 on head matrices; 3% warm-up then cosine; gradient clipping 1.0; options shuffled, score rubrics reversed with probability 0.5; ordinal smoothing 0.1). Targets are the true class; regression uses nine equal-frequency bins of each fit's training rows mapped to Strands' nine ordered levels. Cross-fitting: three label-independent parts per outer training fold; each part is scored by the fit trained on the other two (early stopping on 10% of its training rows, at most 1,000, at most three epochs); test rows average the three fits. Empty text stays missing. Common protocol (Group B): seven fixed predictors, the no-text baseline, 18 datasets, the same five outer folds, fixed headline metrics, datasets as the unit. TAR (E5-small) and the B2 label-trained classifier are context only.

Tables come from `output_JEVPFN/exp10_adaptation/results/group_c/c2_adaptation.zip`.


In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import group_c, study_export
from src.visualize import figures, group_b as gbfig, group_c as gcfig, style
style.apply()
NOTEBOOK = "exp10_adaptation"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. Scope, provenance and the export read by this notebook")
display_json("Notebook execution identity", provenance())
frames, receipt = group_c.read("C2")
print(study_export.describe(receipt))
record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, "Export identity")
WAITING = "Waiting for export: no measurements are shown."

Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\exp10_adaptation\results\group_c\c2_adaptation.zip is absent. No measurements are shown.


## Design and training fits

In [2]:
report_section("2. Design and training fits")
record(group_c.design_counts(), "Group C arms, configurations and C2 fits (plan, not results)")
if not frames:
    print(WAITING)
else:
    display_table("Cross-fitted training fits: epochs kept, validation loss, GPU hours and peak memory", frames["fits"], preview_rows=18)
    record(frames["fits"], "Training fits")
    display_table("Planned versus complete five-fold arms per predictor", frames["coverage"], preview_rows=14)
    record(frames["coverage"], "Coverage")

Waiting for export: no measurements are shown.


## Gains of the four cells

In [3]:
report_section("3. Gains of zero-shot and trained Strands over the no-text baseline")
if not frames:
    print(WAITING)
else:
    gbfig.render(save, gcfig.c2_gains(frames["summary"], name="c2_gains"))
    display_table("Equal-dataset mean gain per arm and predictor", frames["summary"], preview_rows=20)
    record(frames["summary"], "Gain summary")

Waiting for export: no measurements are shown.


## Comparisons

In [4]:
report_section("4. Comparisons: LoRA, head-only and zero-shot; TAR and the label-trained classifier as context")
if not frames:
    print(WAITING)
else:
    summary = frames["contrast_summary"]
    names = sorted(set(summary.representation)) if not summary.empty else []
    main = [n for n in ("lora_vs_zero_shot", "head_only_vs_zero_shot", "lora_vs_head_only", "lora_head_vs_lora_only") if n in names]
    gbfig.render(save, gcfig.contrasts(summary, main, name="c2_cells", what="The 2 x 2 cells against each other"))
    for cell in ("zero_shot", "head_only", "lora_only", "lora_head"):
        context = [n for n in names if n.startswith(f"{cell}_vs_tar/") or n.startswith(f"{cell}_vs_probe/")]
        gbfig.render(save, gcfig.contrasts(summary, context, name=f"c2_context_{cell}", what=f"{gcfig.CELL_LABELS[cell]} Strands against TAR and the label-trained classifier (context)"))
    display_table("Fold-paired contrasts (A minus B), equal-dataset summary", summary, preview_rows=20)
    record(summary, "Contrast summary")

Waiting for export: no measurements are shown.


## Statistical framework (inactive until approved)

Framework application `C2_strands_adaptation` from `config/synthesis/analysis.yaml`. Until the owner sets `framework.status: approved`, this section only prints the awaiting-approval record: no export is read for it and no test is run. After approval it builds the per-dataset learner scores (`src/evaluation/learner_scores.py`), runs `compare_learners` per comparison group (stopping on any missing, incomplete or undefined dataset) and draws one critical-difference diagram per group.

In [5]:
report_section("5. Statistical framework (inactive until the owner approves it)")
from src.evaluation import learner_scores
from src.visualize import critical_difference
from src.visualize.group_b import render as render_figures
framework = learner_scores.run_application("C2_strands_adaptation")
print(f"C2_strands_adaptation: {framework['status']} (framework.status = {framework['framework_status']})")
display_json("Framework application C2_strands_adaptation", learner_scores.summary_record(framework))
render_figures(save, critical_difference.cd_figures(framework))

C2_strands_adaptation: awaiting_owner_approval (framework.status = awaiting_owner_approval)


## Complete report

In [6]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified export was read; no inference, model training or live VSC changes were performed.")

exp10_adaptation

1. Scope, provenance and the export read by this notebook


Notebook execution identity
{
  "code_sha256": "52e3e5b90f1eb41add56bad628ca4189fe3af5d9b55bb3d24a8b3711f5d5f65f",
  "git_head": "b503c624a832ee852c04dd23c07fb4a94c46f73a",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Export identity
{
  "status": "awaiting_export",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\exp10_adaptation\\results\\group_c\\c2_adaptation.zip",
  "created_utc": null,
  "origin": null,
  "sha256": null,
  "bytes": null,
  "schema": null,
  "receipt_s